In [1]:
from infra.spark_utils import build_spark
from infra.tickers_cache import handler_partitions

from pyspark.sql.dataframe import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = build_spark()

file_path = "../data/silver/snapshots/202603_snapshot.csv"

df_silver = spark.read.csv(
    path = file_path,
    sep = ",", 
    header=True,
)

df_silver.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+-----------+------+-------------+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|valor_total|aporte|    exposicao|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+-----------+------+-------------+
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Stock | BERK34 | ...|     stock|              BERK34| 5.0|     120.02|     130.33|     651.65|   0.0|internacional|
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Renda Fixa | Teso...|renda fixa|             tesouro| 1.0|    24000.0|    30000.0|    30000.0|   0.0|     nacional|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | IVVB

In [9]:
def get_variation(df: DataFrame, value_column: str, period_column: str, partition_columns: list[str]) -> DataFrame:
    window = Window.partitionBy(*partition_columns).orderBy(F.col(period_column).asc())
    return (
        df
        .withColumn("previous_value", F.lag(F.col(value_column)).over(window))
        .withColumn(
            "varicao_percentual",
            F.when(
                F.col("previous_value").isNull() | (F.col("previous_value") == 0),
                F.lit(None)
            ).otherwise(
                F.round((((F.col(value_column) - F.col("previous_value")) / F.col("previous_value")) * 100.0), 2)
            )
        )
        .drop("previous_value")
    ) 

In [10]:
path = "../data/gold/202603/202603_gold_home_linha_snapshot.csv" 
df_comp = spark.read.csv(
    path = path,
    sep = ",", 
    header=True,
)

df_comp = get_variation(df_comp, "valor_total", "data_apuracao", partition_columns=["instituicao_fin"])
df_comp.show()

+-------------+----+---+-----------+---------------+------------+-----------+------------------+
|data_apuracao| ano|mes|tipo_escopo|instituicao_fin|nome_metrica|valor_total|varicao_percentual|
+-------------+----+---+-----------+---------------+------------+-----------+------------------+
|   2026-03-14|2026|Mar|       HOME|            ALL| valor_total|   47259.95|              NULL|
|   2026-04-14|2026|Mar|       HOME|            ALL| valor_total|   48259.95|              2.12|
|   2026-05-14|2026|Mar|       HOME|            ALL| valor_total|   49259.95|              2.07|
|   2026-06-14|2026|Mar|       HOME|            ALL| valor_total|   52259.95|              6.09|
|   2026-03-14|2026|Mar|INSTITUICAO|          CLEAR| valor_total|     9008.3|              NULL|
|   2026-04-14|2026|Mar|INSTITUICAO|          CLEAR| valor_total|     9708.3|              7.77|
|   2026-05-14|2026|Mar|INSTITUICAO|          CLEAR| valor_total|     1008.3|            -89.61|
|   2026-06-14|2026|Mar|INSTIT

In [6]:
# dfs graficos de linha de comparacao, de barra e valores atuais para botoes HOME
df_instituicao = (
    df_silver
    .groupBy("data_apuracao","ano", "mes", "instituicao_fin")
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("mes"),
        F.lit("INSTITUICAO").alias("tipo_escopo"),
        F.col("instituicao_fin"),
        F.lit("valor_total").alias("nome_metrica"),
        F.col("valor_total"),
    )
)

df_total = (
    df_silver
    .groupBy("data_apuracao", "ano", "mes")
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("mes"),
        F.col("valor_total"),
        F.lit("HOME").alias("tipo_escopo"),
        F.lit("ALL").alias("instituicao_fin"),
        F.lit("valor_total").alias("nome_metrica"),
    )
)

df_evolucao_ano_mes = df_instituicao.unionByName(df_total)

df_evolucao_ano_mes = df_evolucao_ano_mes.withColumn("valor_total", F.round(F.col("valor_total"), 2)) # garantia de 2 casas

df_evolucao_ano_mes = get_variation(df_evolucao_ano_mes, "valor_total", "data_apuracao", partition_columns=["instituicao_fin"])

# grafico de linha 
df_evolucao_ano_mes.show()

print(handler_partitions(df_evolucao_ano_mes, 'gold', 'home_linha'))


latest_value_window = Window.partitionBy("instituicao_fin").orderBy(F.col("data_apuracao").desc())

df_val_atual = (
    df_evolucao_ano_mes
    .withColumn("rn", F.row_number().over(latest_value_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# botoes
df_val_atual.show()
print(handler_partitions(df_val_atual, 'gold', 'home_botoes'))

df_evolucao_ano = (
    df_evolucao_ano_mes
    .groupBy("data_apuracao","ano", "instituicao_fin")
    .agg(F.sum("valor_total").alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("instituicao_fin"),
        F.round(F.col("valor_total"), 2).alias("valor_total"),
        F.lit("HOME").alias("tipo_escopo"),
        F.lit("valor_total_anual").alias("nome_metrica"),
    )
)

# grafico de barras
df_evolucao_ano.show()
print(handler_partitions(df_evolucao_ano, 'gold', 'home_barras'))

+-------------+----+---+-----------+---------------+------------+-----------+------------------+
|data_apuracao| ano|mes|tipo_escopo|instituicao_fin|nome_metrica|valor_total|varicao_percentual|
+-------------+----+---+-----------+---------------+------------+-----------+------------------+
|   2026-03-14|2026|Mar|       HOME|            ALL| valor_total|   47259.95|              NULL|
|   2026-03-14|2026|Mar|INSTITUICAO|          CLEAR| valor_total|     9008.3|              NULL|
|   2026-03-14|2026|Mar|INSTITUICAO|         NUBANK| valor_total|     7600.0|              NULL|
|   2026-03-14|2026|Mar|INSTITUICAO|             XP| valor_total|   30651.65|              NULL|
+-------------+----+---+-----------+---------------+------------+-----------+------------------+

Success file saved at: C:\desenvolvimento\investment-visualizer\data\gold\202603\202603_gold_home_linha_snapshot.csv
+-------------+----+---+-----------+---------------+------------+-----------+------------------+
|data_apu

In [6]:
# dfs graficos de pizza HOME e INSTITUICAO
latest_value_window_name = Window.partitionBy("instituicao_fin", "tipo", "nome").orderBy(F.col("data_apuracao").desc())

df_dados_atuais = (
    df_silver
    .withColumn("rn", F.row_number().over(latest_value_window_name))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_exposicao_all = (
    df_dados_atuais
    .groupBy('data_apuracao', 'exposicao')
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col('data_apuracao')
        , F.col('exposicao')
        , F.col("valor_total")
        , F.lit("ALL").alias("instituicao_fin")
    )
)

df_exposicao_instituicao = (
    df_dados_atuais
    .groupBy('data_apuracao', 'exposicao', 'instituicao_fin')
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col('data_apuracao')
        , F.col('instituicao_fin')
        , F.col('exposicao')
        , F.col("valor_total")
    )
)

df_exposicao = df_exposicao_all.unionByName(df_exposicao_instituicao)

# grafico pizza exposicao HOME e INSTITUTICAO
df_exposicao.show()
print(handler_partitions(df_exposicao, 'gold', 'pizza_expo'))

df_tipo_all = (
    df_dados_atuais
    .groupBy('data_apuracao', 'tipo')
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col('data_apuracao')
        , F.col('tipo')
        , F.col("valor_total")
        , F.lit("ALL").alias("instituicao_fin")
    )
)

df_tipo_instituicao = (
    df_dados_atuais
    .groupBy('data_apuracao', 'tipo', 'instituicao_fin')
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col('data_apuracao')
        , F.col('instituicao_fin')
        , F.col('tipo')
        , F.col("valor_total")
    )
)

df_tipo = df_tipo_all.unionByName(df_tipo_instituicao)
df_tipo = df_tipo.withColumn('valor_total', F.round(F.col("valor_total"), 2))

# grafico pizza tipo HOME e INSTITUTICAO
df_tipo.show()
print(handler_partitions(df_tipo, 'gold', 'pizza_tipo'))

+-------------+-------------+-----------+---------------+
|data_apuracao|    exposicao|valor_total|instituicao_fin|
+-------------+-------------+-----------+---------------+
|   2026-03-14|internacional|    7914.45|            ALL|
|   2026-03-14|     nacional|    39345.5|            ALL|
|   2026-03-14|internacional|     7262.8|          CLEAR|
|   2026-03-14|     nacional|     1745.5|          CLEAR|
|   2026-03-14|     nacional|     7600.0|         NUBANK|
|   2026-03-14|     nacional|    30000.0|             XP|
|   2026-03-14|internacional|     651.65|             XP|
+-------------+-------------+-----------+---------------+

Success file saved at: C:\desenvolvimento\investment-visualizer\data\gold\202603\202603_gold_pizza_expo_snapshot.csv
+-------------+----------+-----------+---------------+
|data_apuracao|      tipo|valor_total|instituicao_fin|
+-------------+----------+-----------+---------------+
|   2026-03-14|     stock|    9659.95|            ALL|
|   2026-03-14|renda fix

In [7]:
# dfs graficos de linha de comparacao, de barra e valores atuais INSTITUICAO
df_instituicao_page = (
    df_silver
    .groupBy("data_apuracao","ano", "mes", "instituicao_fin", 'nome')
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("mes"),
        F.col("instituicao_fin"),
        F.col("nome"),
        F.col("valor_total"),
        F.lit("valor_total").alias("nome_metrica"),
        F.lit("INSTITUICAO").alias("tipo_escopo"),
    )
)

df_total_page = (
    df_silver
    .groupBy("data_apuracao", "ano", "mes", 'instituicao_fin')
    .agg(F.sum(F.col("valor_total")).alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("mes"),
        F.col("instituicao_fin"),
        F.lit('ALL').alias('nome'),
        F.col("valor_total"),
        F.lit("valor_total").alias("nome_metrica"),
        F.lit("INSTITUICAO").alias("tipo_escopo"),
    )
)


df_evolucao_ano_mes_page = df_instituicao_page.unionByName(df_total_page)
df_evolucao_ano_mes_page = df_evolucao_ano_mes_page.withColumn("valor_total", F.round(F.col("valor_total"), 2)) # garantia de 2 casas 

# grafico de linha 
df_evolucao_ano_mes_page.show()
print(handler_partitions(df_evolucao_ano_mes_page, 'gold', 'instituicao_linha'))


latest_value_window = Window.partitionBy("instituicao_fin", "nome").orderBy(F.col("data_apuracao").desc())

df_val_atual_page = (
    df_evolucao_ano_mes_page
    .withColumn("rn", F.row_number().over(latest_value_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# labels
df_val_atual_page.show()
print(handler_partitions(df_val_atual_page, 'gold', 'instituicao_label'))

df_evolucao_ano_page = (
    df_evolucao_ano_mes_page
    .groupBy("data_apuracao","ano", "instituicao_fin", "nome")
    .agg(F.sum("valor_total").alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("instituicao_fin"),
        F.col("nome"),
        F.round(F.col("valor_total"), 2).alias("valor_total"),
        F.lit("valor_total_anual").alias("nome_metrica"),
        F.lit("HOME").alias("tipo_escopo"),
    )
)

# grafico de barras
df_evolucao_ano.show()
print(handler_partitions(df_val_atual_page, 'gold', 'instituicao_barras'))

+-------------+----+---+---------------+--------------------+-----------+------------+-----------+
|data_apuracao| ano|mes|instituicao_fin|                nome|valor_total|nome_metrica|tipo_escopo|
+-------------+----+---+---------------+--------------------+-----------+------------+-----------+
|   2026-03-14|2026|Mar|             XP|              BERK34|     651.65| valor_total|INSTITUICAO|
|   2026-03-14|2026|Mar|             XP|             tesouro|    30000.0| valor_total|INSTITUICAO|
|   2026-03-14|2026|Mar|          CLEAR|              IVVB11|     5959.5| valor_total|INSTITUICAO|
|   2026-03-14|2026|Mar|          CLEAR|              BOVA11|     1745.5| valor_total|INSTITUICAO|
|   2026-03-14|2026|Mar|          CLEAR|              BERK34|     1303.3| valor_total|INSTITUICAO|
|   2026-03-14|2026|Mar|         NUBANK|reserva de emerge...|     7600.0| valor_total|INSTITUICAO|
|   2026-03-14|2026|Mar|             XP|                 ALL|   30651.65| valor_total|INSTITUICAO|
|   2026-0